# Optimize Tensorflow Pipeline Performance: prefetch & cache


In [1]:
import tensorflow as tf
import time

In [2]:
tf.__version__

'2.21.0'

In [3]:
class FileDataset(tf.data.Dataset):
    def read_files_in_batch(number_of_samples):
        # open files
        time.sleep(0.03)
        for sample_index in range(number_of_samples):
            time.sleep(0.015)
            yield (sample_index,)

    def __new__(cls, num_samples=3):
        return tf.data.Dataset.from_generator(
            cls.read_files_in_batch,
            output_signature=(tf.TensorSpec(shape=(1,), dtype=tf.int64)),
            args=(num_samples,),
        )

In [20]:
def benchmark(dataset, num_epochs=2):
    for epoch_num in range(num_epochs):
        for sample in dataset:
            time.sleep(0.01)

In [5]:
%%timeit
benchmark(FileDataset(), num_epochs=2)

251 ms ± 6.65 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
benchmark(FileDataset().prefetch(1), num_epochs=2)

250 ms ± 15 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
%%timeit
benchmark(FileDataset().prefetch(tf.data.AUTOTUNE), num_epochs=2)

241 ms ± 3.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
dataset = tf.data.Dataset.range(5)
for d in dataset:
    print(d)

tf.Tensor(0, shape=(), dtype=int64)
tf.Tensor(1, shape=(), dtype=int64)
tf.Tensor(2, shape=(), dtype=int64)
tf.Tensor(3, shape=(), dtype=int64)
tf.Tensor(4, shape=(), dtype=int64)


In [10]:
dataset = dataset.map(lambda x: x**2)
for d in dataset:
    print(d)

tf.Tensor(0, shape=(), dtype=int64)
tf.Tensor(1, shape=(), dtype=int64)
tf.Tensor(4, shape=(), dtype=int64)
tf.Tensor(9, shape=(), dtype=int64)
tf.Tensor(16, shape=(), dtype=int64)


In [11]:
dataset = dataset.cache()
for d in dataset.as_numpy_iterator():
    print(d)

0
1
4
9
16


In [12]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [13]:
list(dataset.as_numpy_iterator())

[np.int64(0), np.int64(1), np.int64(4), np.int64(9), np.int64(16)]

In [24]:
def mapped_function(s):
    tf.py_function(lambda: time.sleep(0.03), [], ())
    return s

In [37]:
%%timeit -n1 -r1
benchmark(FileDataset().map(mapped_function),5)

1.1 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [41]:
%%timeit -n1 -r1
benchmark(FileDataset().map(mapped_function).cache(),5)

399 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
